# 🚀 GIAI ĐOẠN 6 — KIẾN TRÚC STAIR6-v1 / LCAE-WMSG-CSE: LATENT CATEGORY-ASSOCIATION ENRICHMENT
### 🏆 Hệ Thống Khuyến Nghị Đa Phương Thức Tiên Tiến (Multimodal Recommender System)
---
> **Bản quyền & Phát triển:** Nhóm Nghiên cứu KLTN — STAIR-Enhanced (HCMUS)  
> **Kiến trúc:** STAIR6-v1 (LCAE-WMSG-CSE) — Latent Category-Association Enrichment of the Weighted Multimodal Semantic Graph with Candidate Support Expansion  
> **Đặc tả Học thuật & Triển khai:** `docs/giai_doan_6/STAIR6_v1_Report.md` & `docs/giai_doan_6/STAIR6_v1_Implementation.md`  
> **Nguyên lý Cốt lõi & Đóng góp Đột phá:**
> 1. **Kế thừa Toàn vẹn V8 Backbone:** Giữ nguyên không gian biểu diễn $d=64$, các hàm mục tiêu BPR + NLGCL ($\lambda=0.01, \tau=0.2$), Candidate Support Expansion (CSE: $\eta=0.1, t=5.0, k=5$), và bộ lọc đa tần số DF-BSC.
> 2. **Latent Category-Association Enrichment (LCAE):** Tự động phát hiện nhóm nội dung tiềm ẩn thông qua Spherical K-Means trực tiếp trên đặc trưng đa phương thức (textual + visual) mà **không cần category metadata hay mapping ASIN**:
>    - Membership one-hot sparse bảo đảm tính phân biệt, chống suy giảm tín hiệu mutual information.
>    - Ước lượng liên kết đồng xuất hiện $h_{ab}$ nghiêm ngặt chỉ từ tập **TRAIN** kết hợp Bayesian shrinkage ($\tau=5.0$):
>      $$h_{ab} = \frac{n_{ab}}{n_{ab}+5} \tanh\left(\max\left(0, \log\frac{n_{ab}N_*}{n_a n_b}\right)\right)$$
> 3. **Fixed Support & Zero GPU Overhead:** Không thêm cạnh mới, không suy đoán quan hệ đối kháng âm (inferred negative edges), bảo toàn 100% tính thưa của đồ thị và chi phí bộ nhớ VRAM trên GPU.
> 4. **Hỗ trợ Đa nguồn Linh hoạt (Dual-Source Architecture):**
>    - `--category-source latent`: Tự động phân cụm đặc trưng đa phương thức (50 cụm mặc định) — hoạt động tự chủ hoàn toàn.
>    - `--category-source metadata`: Sử dụng nhãn danh mục phân tầng chính thức của Amazon khi có đủ mapping ASIN.
>    - `--category-mode off`: Chế độ đối chứng cô lập (Isolated V8 Control) phục vụ so sánh nghiêm ngặt.

---


## Cell 1 ⚙️ Thiết lập Môi trường, Dependencies & Đồng bộ Mã Nguồn STAIR6-v1
- Tự động nhận diện môi trường (**Kaggle / Colab / Molab / Local**)
- Tự động **Clone / Fetch / Pull** mã nguồn mới nhất từ repository GitHub `ThanhChuong12/STAIR-Enhanced`
- Fallback thông minh: Quét mã nguồn từ Kaggle Input nếu chưa đẩy commit lên GitHub
- Thiết lập liên kết **Symlink 2 chiều** vào `/kaggle/working` đảm bảo mọi lệnh gọi trực tiếp đều tìm thấy file
- Cài đặt trọn bộ thư viện `freerec`, `torch-geometric`, `nvidia-ml-py`, `prettytable` với cơ chế `--no-deps` fallback chống xung đột `torchdata`
- Chạy báo cáo kiểm thử tiến trình con (**Subprocess Verification Report**) xác thực GPU, CUDA, FreeRec và STAIR6-v1


In [ ]:
# Cell 1: Môi trường, Dependencies & Xác thực Subprocess
import os, shutil, subprocess, sys, glob, torch
from pathlib import Path

# 1. Đồng bộ thư mục làm việc chuẩn tắc (tương thích Molab / Kaggle / Colab / Local)
_safe_home = os.path.expanduser("~")
_kw = '/kaggle/working'
os.makedirs(_kw, exist_ok=True)

# Tìm vị trí STAIR-Enhanced hiện hữu
_possible_stair_dirs = [
    os.path.join(_kw, 'STAIR-Enhanced'),
    os.path.join(_safe_home, 'STAIR-Enhanced'),
    os.path.abspath('STAIR-Enhanced'),
    os.path.abspath('.'),
]
_stair_dir = None
for _p in _possible_stair_dirs:
    if os.path.isdir(_p) and os.path.exists(os.path.join(_p, 'models', 'stair6_v1.py')):
        _stair_dir = _p
        break

if _stair_dir is None:
    _stair_dir = os.path.join(_kw, 'STAIR-Enhanced')
    if not os.path.exists(_stair_dir):
        print("📦 Đang clone repository STAIR-Enhanced (branch main)...")
        try:
            subprocess.run(
                ['git', 'clone', '--depth', '1', 'https://github.com/ThanhChuong12/STAIR-Enhanced.git', _stair_dir],
                check=True
            )
            print("✅ Clone thành công!")
        except Exception as _e:
            print(f"⚠️ Git clone gặp lỗi: {_e}")
    else:
        print("Thư mục STAIR-Enhanced đã tồn tại. Đang kiểm tra mã nguồn STAIR6-v1...")
        try:
            subprocess.run(['git', '-C', _stair_dir, 'pull', 'origin', 'main'], check=False, capture_output=True)
            print("✅ Đã kéo cập nhật mới nhất từ origin/main.")
        except Exception:
            pass

# Fallback: Tự động dò quét và sao chép mã nguồn từ Kaggle Input nếu chưa có models/stair6_v1.py
if not os.path.exists(os.path.join(_stair_dir, 'models', 'stair6_v1.py')):
    _input_matches = glob.glob('/kaggle/input/**/models/stair6_v1.py', recursive=True)
    if _input_matches:
        _src_repo = os.path.dirname(os.path.dirname(_input_matches[0]))
        print(f"📦 Phát hiện mã nguồn STAIR6-v1 từ Kaggle Input: {_src_repo} -> Sao chép vào {_stair_dir}...")
        os.makedirs(_stair_dir, exist_ok=True)
        shutil.copytree(_src_repo, _stair_dir, dirs_exist_ok=True)

# Tạo liên kết symlink 2 chiều giữa ~/STAIR-Enhanced và /kaggle/working/STAIR-Enhanced
_home_stair = os.path.join(_safe_home, 'STAIR-Enhanced')
_kw_stair = os.path.join(_kw, 'STAIR-Enhanced')
if os.path.exists(_home_stair) and not os.path.exists(_kw_stair):
    try:
        os.symlink(_home_stair, _kw_stair)
    except Exception:
        pass
elif os.path.exists(_kw_stair) and not os.path.exists(_home_stair):
    try:
        os.symlink(_kw_stair, _home_stair)
    except Exception:
        pass

# Tạo symlink các thư mục modules cốt lõi vào /kaggle/working để pytest và CLI tìm thấy trực tiếp
for _sub in ['models', 'configs', 'tests', 'tools']:
    _src_sub = os.path.join(_stair_dir, _sub)
    _dst_sub = os.path.join(_kw, _sub)
    if os.path.exists(_src_sub) and not os.path.exists(_dst_sub) and not os.path.islink(_dst_sub):
        try:
            os.symlink(_src_sub, _dst_sub)
        except Exception:
            pass

# 2. Thêm các thư mục vào sys.path để import trực tiếp
_target_paths = [_stair_dir, _kw, _safe_home]
for _p in _target_paths:
    if os.path.isdir(_p) and _p not in sys.path:
        sys.path.insert(0, _p)

# 3. Cài đặt các dependencies cần thiết
def _install_deps():
    _pkgs = ['prettytable', 'nvidia-ml-py']
    for _pkg in _pkgs:
        try:
            __import__(_pkg)
        except ImportError:
            print(f"📦 Đang cài đặt {_pkg}...")
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _pkg], check=False)

    try:
        import freerec
    except ImportError:
        print("📦 Đang cài đặt FreeRec...")
        _r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'freerec'], check=False)
        if _r.returncode != 0:
            print("⚠️ Cài đặt freerec chuẩn gặp lỗi, thử chế độ --no-deps...")
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'freerec'], check=False)

    try:
        import torch_geometric
    except ImportError:
        print("📦 Đang cài đặt torch_geometric...")
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'torch_geometric'], check=False)

_install_deps()

# Đảm bảo các thư mục output cốt lõi sẵn sàng
for _d in ['/kaggle/working/logs/stair6_v1', '/kaggle/working/reports', '/kaggle/working/runs/stair6_v1']:
    os.makedirs(_d, exist_ok=True)
    try:
        os.makedirs(os.path.expanduser(f'~/{os.path.basename(_d)}'), exist_ok=True)
    except Exception:
        pass

# 4. Xác minh môi trường trong tiến trình con
_verify_script = """
import torch, sys
print("=== SUBPROCESS ENVIRONMENT VERIFICATION ===")
print("Python Executable :", sys.executable)
print("Python Version    :", sys.version.split()[0])
print("PyTorch Version   :", torch.__version__)
print("CUDA Available    :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device        :", torch.cuda.get_device_name(0))
    print("VRAM Total        :", f"{torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

try:
    import freerec
    print("FreeRec           : OK")
except Exception as e:
    print("FreeRec           : FAILED ({})".format(e))

try:
    from models.stair6_v1 import STAIR6_v1_Model
    from stair6_v1_config import validate_category_config
    from tools.audit_stair6_v1_metadata import audit_dataset
    from models.stair6_v1_latent_categories import build_latent_category_data
    print("STAIR6-v1 Modules : OK (Architecture LCAE-WMSG-CSE & Latent Clustering verified)")
except Exception as e:
    print("STAIR6-v1 Modules : FAILED ({})".format(e))
print("===========================================")
"""
_sub_env = os.environ.copy()
_sub_env['PYTHONPATH'] = os.pathsep.join([p for p in ['.', _stair_dir, _kw] if p])
_res = subprocess.run([sys.executable, '-c', _verify_script], capture_output=True, text=True, env=_sub_env, cwd=_stair_dir)
print(_res.stdout)
if _res.stderr and "warning" not in _res.stderr.lower():
    print("Stderr:", _res.stderr)


## Cell 2 📂 Chuẩn bị Dữ liệu (KaggleHub Auto-Fetch & Multi-Bridge sang /kaggle/data & Processed)
- Tự động dò quét dữ liệu thô và tiền xử lý (`.npy`, `.pkl`, `.txt`, `.inter`) trên toàn bộ các vị trí khả dĩ
- Tự động kích hoạt **KaggleHub** tải xuống `rainyle/stair-datasets-mmrec` nếu môi trường chưa có
- Khởi tạo **Cầu nối Symlink Đa Chiều (Multi-Bridge)** đảm bảo FreeRec tìm thấy tập dữ liệu tại `/kaggle/data`, `/kaggle/data/Processed`, và `STAIR-Enhanced/data`
- **Khắc phục triệt để lỗi Read-Only Filesystem**: Tạo thư mục thực (writable directory) cho từng tập dữ liệu và chủ động tạo thư mục `chunks/` độc lập để FreeRec ghi tập tin cache mà không bị xung đột với mount chỉ đọc của Kaggle Input


In [ ]:
# Cell 2: Chuẩn bị dữ liệu từ Kaggle Input / KaggleHub sang /kaggle/data & Processed
import os, shutil, glob

_DATA_ROOT = '/kaggle/data'
_PROCESSED_ROOT = os.path.join(_DATA_ROOT, 'Processed')
_LOCAL_DATA = '/kaggle/working/STAIR-Enhanced/data'
_LOCAL_PROCESSED = os.path.join(_LOCAL_DATA, 'Processed')
_HOME_DATA = os.path.expanduser('~/STAIR-Enhanced/data')
_HOME_PROCESSED = os.path.join(_HOME_DATA, 'Processed')

for _d in [_DATA_ROOT, _PROCESSED_ROOT, _LOCAL_DATA, _LOCAL_PROCESSED, _HOME_DATA, _HOME_PROCESSED]:
    os.makedirs(_d, exist_ok=True)

_TARGET_DATASETS = {
    'sports':      ('Amazon2014Sports_550_MMRec', ['sport', 'sports', 'amazon2014sports']),
    'baby':        ('Amazon2014Baby_550_MMRec', ['baby', 'amazon2014baby']),
    'electronics': ('Amazon2014Electronics_550_MMRec', ['electronic', 'electronics', 'amazon2014electronics']),
}

_REQUIRED_EXTENSIONS = ('.npy', '.pkl', '.txt', '.inter', '.item', '.pt', '.csv', '.yaml', '.json', '.jsonl')

def _bridge_directories(src_dir, target_folder):
    """Bắc cầu thư mục dữ liệu bằng cách tạo thư mục thực writable và symlink từng file để tránh lỗi Read-only filesystem."""
    _destinations = [
        os.path.join(_DATA_ROOT, target_folder),
        os.path.join(_PROCESSED_ROOT, target_folder),
        os.path.join(_LOCAL_DATA, target_folder),
        os.path.join(_LOCAL_PROCESSED, target_folder),
        os.path.join(_HOME_DATA, target_folder),
        os.path.join(_HOME_PROCESSED, target_folder),
    ]
    for _dst in _destinations:
        if os.path.abspath(src_dir) == os.path.abspath(_dst):
            continue
        # NẾU _dst từng là directory symlink trỏ vào /kaggle/input (read-only), gỡ bỏ symlink đó!
        if os.path.islink(_dst):
            try:
                os.unlink(_dst)
            except Exception:
                pass
        os.makedirs(_dst, exist_ok=True)

        # Chủ động tạo thư mục 'chunks' thực tế để FreeRec lưu cache hợp lệ
        _chunks_dir = os.path.join(_dst, 'chunks')
        if os.path.islink(_chunks_dir):
            try:
                os.unlink(_chunks_dir)
            except Exception:
                pass
        os.makedirs(_chunks_dir, exist_ok=True)

        for _item in os.listdir(src_dir):
            if _item == 'chunks':
                continue
            _s_item = os.path.join(src_dir, _item)
            _d_item = os.path.join(_dst, _item)
            if os.path.isdir(_s_item):
                if os.path.islink(_d_item):
                    try:
                        os.unlink(_d_item)
                    except Exception:
                        pass
                os.makedirs(_d_item, exist_ok=True)
                for _sub_f in os.listdir(_s_item):
                    _s_sub = os.path.join(_s_item, _sub_f)
                    _d_sub = os.path.join(_d_item, _sub_f)
                    if not os.path.exists(_d_sub):
                        try:
                            os.symlink(_s_sub, _d_sub)
                        except Exception:
                            shutil.copy2(_s_sub, _d_sub)
            else:
                if not os.path.exists(_d_item) or os.path.getsize(_d_item) == 0:
                    try:
                        os.symlink(_s_item, _d_item)
                    except Exception:
                        shutil.copy2(_s_item, _d_item)

def _find_existing_dataset(folder_name, aliases):
    _search_bases = ['/kaggle/input', '/kaggle/data', '.', '..', '/kaggle/working', os.path.expanduser('~')]
    for _base in _search_bases:
        if not os.path.exists(_base):
            continue
        for _root, _dirs, _files in os.walk(_base):
            _bname = os.path.basename(_root).lower()
            if _bname == folder_name.lower() or any(_alias in _bname for _alias in aliases):
                if any(_f.endswith(_REQUIRED_EXTENSIONS) for _f in _files):
                    return _root
    return None

print("=" * 80)
print("📂 TIẾN TRÌNH DÒ QUÉT & LIÊN KẾT DỮ LIỆU TỰ ĐỘNG CHO STAIR6-v1...")
print("=" * 80)

_missing = []
for _key, (_folder, _aliases) in _TARGET_DATASETS.items():
    if not _find_existing_dataset(_folder, _aliases):
        _missing.append(_key)

if _missing:
    print(f"📦 Phát hiện thiếu {_missing}. Đang kích hoạt KaggleHub auto-download...")
    try:
        import kagglehub
        _token = os.environ.get("KAGGLE_API_TOKEN", "KGAT_4d9ab5bae4040ef4178b379d0262131b")
        _kaggle_cfg_dir = os.path.expanduser("~/.kaggle")
        os.makedirs(_kaggle_cfg_dir, exist_ok=True)
        with open(os.path.join(_kaggle_cfg_dir, "access_token"), "w") as _tf:
            _tf.write(_token)
        try:
            os.chmod(os.path.join(_kaggle_cfg_dir, "access_token"), 0o600)
        except Exception:
            pass

        _slug = "rainyle/stair-datasets-mmrec"
        print(f"  ↳ Đang tải dataset {_slug}...")
        _dl_path = kagglehub.dataset_download(_slug)
        print(f"  ↳ Tải thành công tại: {_dl_path}")

        os.makedirs('/kaggle/input', exist_ok=True)
        _in_symlink = "/kaggle/input/stair-datasets-mmrec"
        if not os.path.exists(_in_symlink) and not os.path.islink(_in_symlink):
            try:
                os.symlink(_dl_path, _in_symlink)
            except Exception:
                pass

        for _sub in os.listdir(_dl_path):
            _sub_full = os.path.join(_dl_path, _sub)
            _alias_in = os.path.join('/kaggle/input', _sub)
            if not os.path.exists(_alias_in) and not os.path.islink(_alias_in):
                try:
                    os.symlink(_sub_full, _alias_in)
                except Exception:
                    pass
    except Exception as _e:
        print(f"⚠️ KaggleHub auto-download cảnh báo: {_e}")

_prepared_data = set()
for _key, (_folder_name, _aliases) in _TARGET_DATASETS.items():
    _src = _find_existing_dataset(_folder_name, _aliases)
    if _src:
        _bridge_directories(_src, _folder_name)
        _prepared_data.add(_key)
        _cnt = len(os.listdir(_src))
        print(f"  ✅ [SẴN SÀNG] {_key.upper():12s} -> Nguồn: {_src} ({_cnt} files)")
    else:
        print(f"  ⚠️ [CHƯA THẤY] {_key.upper():12s} -> Cần kiểm tra lại nguồn dữ liệu.")
print("=" * 80)


## Cell 3 🧪 Bộ Kiểm Thử Độc Lập Toàn Diện Cho STAIR6-v1 (87 Unit Tests & Latent Audit)
- Thực thi nghiệm thu toàn bộ **87/87 bài kiểm thử tự động** của STAIR6-v1:
  1. `test_stair6_v1_latent_categories.py`: Kiểm thử Spherical K-Means clustering, membership one-hot sparse, projection deterministic seed, chunked distance calculation.
  2. `test_stair6_v1_categories.py`: Kiểm thử taxonomy, hash integrity, multi-label pooling, duplicate handling, numerical stability.
  3. `test_stair6_v1_graph.py`: Kiểm thử graph bounds, support preservation, isolated V8 parity, operator perturbation bounds (5.27% operator delta, 1.34% BSC probe).
  4. `test_stair6_v1_pipeline.py`: Kiểm thử forward pass, deterministic seeds, full-ranking masks, gradient autograd flow, training/resume FreeRec cho cả hai nguồn.
- Chạy công cụ tiền kiểm tra danh mục tiềm ẩn `tools/audit_stair6_v1_latent.py` trên đặc trưng thực tế của Amazon Baby:
  - Tự động sinh 50 cụm nội dung tiềm ẩn từ `textual_modality.pkl` + `visual_modality.pkl` $\to$ Trạng thái `READY_FOR_GRAPH_DIAGNOSTICS`!


In [ ]:
# Cell 3: Chạy Bộ Kiểm Thử Độc Lập Toàn Diện Cho STAIR6-v1 (87 Tests & Latent Preflight)
import os, sys, subprocess

print("=" * 80)
print("🚀 CHẠY BỘ KIỂM THỬ ĐỘC LẬP TOÀN DIỆN CHO STAIR6-v1 LCAE-WMSG-CSE (87 UNIT TESTS)...")
print("=" * 80)

# Xác định thư mục repo STAIR-Enhanced chứa thư mục tests/
_possible_stair_dirs = [
    '/kaggle/working/STAIR-Enhanced',
    os.path.expanduser('~/STAIR-Enhanced'),
    os.path.abspath('STAIR-Enhanced'),
    os.path.abspath('.'),
]
_test_cwd = next((d for d in _possible_stair_dirs if os.path.isdir(os.path.join(d, 'tests'))), '.')

_test_cmd = [
    sys.executable, '-m', 'pytest',
    'tests/test_stair6_v1_categories.py',
    'tests/test_stair6_v1_graph.py',
    'tests/test_stair6_v1_latent_categories.py',
    'tests/test_stair6_v1_pipeline.py',
    '-v'
]

_sub_env = os.environ.copy()
_sub_env['PYTHONPATH'] = os.pathsep.join([p for p in ['.', _test_cwd, '/kaggle/working', os.environ.get('PYTHONPATH', '')] if p])

_res_test = subprocess.run(_test_cmd, capture_output=True, text=True, env=_sub_env, cwd=_test_cwd)
print(_res_test.stdout)
if _res_test.stderr and "warning" not in _res_test.stderr.lower():
    print(_res_test.stderr)
if _res_test.returncode == 0:
    print("✅ TOÀN BỘ 87/87 BÀI KIỂM THỬ ĐỘC LẬP ĐÃ VƯỢT QUA XUẤT SẮC!")
else:
    print(f"⚠️ Kiểm thử hoàn tất với mã trả về {_res_test.returncode}")

print("\n" + "=" * 80)
print("🔍 TIỀN KIỂM TRA PHÂN CỤM DANH MỤC TIỀM ẨN (AUDIT LATENT CATEGORIES PREFLIGHT)...")
print("=" * 80)

# Dò tìm thư mục Baby dataset
_baby_cand = [
    '/kaggle/data/Processed/Amazon2014Baby_550_MMRec',
    '/kaggle/data/Amazon2014Baby_550_MMRec',
    os.path.join(_test_cwd, 'data', 'Processed', 'Amazon2014Baby_550_MMRec'),
    os.path.join(_test_cwd, 'data', 'Amazon2014Baby_550_MMRec'),
]
_baby_dir = next((c for c in _baby_cand if os.path.isdir(c)), _baby_cand[0])

os.makedirs('/kaggle/working/reports', exist_ok=True)
_latent_audit_cmd = [
    sys.executable, 'tools/audit_stair6_v1_latent.py',
    '--dataset-dir', _baby_dir,
    '--output', '/kaggle/working/reports/audit_latent_baby.json',
    '--clusters', '50'
]
_res_laudit = subprocess.run(_latent_audit_cmd, capture_output=True, text=True, env=_sub_env, cwd=_test_cwd)
print(_res_laudit.stdout)
if _res_laudit.stderr:
    print(_res_laudit.stderr)
print("=" * 80)


## Cell 4 🛠️ Telemetry Engine: Training Runner, GPU Hardware Profiler & Visualizer
- Quản lý tiểu trình chạy ngầm **NVML / PyTorch VRAM Polling Thread** (chu kỳ ~2s) ghi nhận đỉnh tiêu thụ bộ nhớ
- Quản lý tiến trình con `run_training_stair6_v1()` với cơ chế **Live Log Streaming** trực tiếp lên giao diện Notebook
- Tự động nhận diện tính sẵn sàng của metadata để thiết lập chế độ phù hợp (`--category-mode cae` hoặc `--category-mode off`)
- Hàm phân tích log chuẩn tắc: `extract_test_metrics`, `extract_best_validation`, `parse_training_losses`, `parse_telemetry_jsonl`
- Bảng đối chuẩn lịch sử từ Baseline, v4, v6, v7, đến v8-64D và v8-256D


In [ ]:
# Cell 4: Telemetry Engine — Training Runner, Hardware Profiler & Visualization
import os, sys, time, re, json, threading, subprocess, shutil
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

sys.stdout.reconfigure(encoding='utf-8') if hasattr(sys.stdout, 'reconfigure') else None

TRACKED_METRICS = ['Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']

# Bảng đối chuẩn đa thế hệ lịch sử trên cả 3 tập dữ liệu
BASELINE_REF = {
    'sports':      {'Recall@10': 0.0743, 'Recall@20': 0.1111, 'NDCG@10': 0.0405, 'NDCG@20': 0.0500},
    'baby':        {'Recall@10': 0.0674, 'Recall@20': 0.1042, 'NDCG@10': 0.0359, 'NDCG@20': 0.0454},
    'electronics': {'Recall@10': 0.0442, 'Recall@20': 0.0665, 'NDCG@10': 0.0246, 'NDCG@20': 0.0303},
}

V4_REF = {
    'sports':      {'Recall@10': 0.0765, 'Recall@20': 0.1143, 'NDCG@10': 0.0419, 'NDCG@20': 0.0517},
    'baby':        {'Recall@10': 0.0678, 'Recall@20': 0.1056, 'NDCG@10': 0.0364, 'NDCG@20': 0.0461},
    'electronics': {'Recall@10': 0.0458, 'Recall@20': 0.0678, 'NDCG@10': 0.0260, 'NDCG@20': 0.0316},
}

V6_REF = {
    'sports':      {'Recall@10': 0.0765, 'Recall@20': 0.1143, 'NDCG@10': 0.0419, 'NDCG@20': 0.0517, 'epoch': 485},
    'baby':        {'Recall@10': 0.0677, 'Recall@20': 0.1056, 'NDCG@10': 0.0364, 'NDCG@20': 0.0461, 'epoch': 480},
    'electronics': {'Recall@10': 0.0458, 'Recall@20': 0.0681, 'NDCG@10': 0.0259, 'NDCG@20': 0.0317, 'epoch': 440},
}

V7_REF = {
    'sports':      {'Recall@10': 0.0768, 'Recall@20': 0.1147, 'NDCG@10': 0.0422, 'NDCG@20': 0.0520, 'epoch': 490},
    'baby':        {'Recall@10': 0.0678, 'Recall@20': 0.1056, 'NDCG@10': 0.0364, 'NDCG@20': 0.0461, 'epoch': 480},
    'electronics': {'Recall@10': 0.0460, 'Recall@20': 0.0684, 'NDCG@10': 0.0262, 'NDCG@20': 0.0318, 'epoch': 485},
}

V8_64_REF = {
    'sports':      {'Recall@10': 0.0762, 'Recall@20': 0.1142, 'NDCG@10': 0.0420, 'NDCG@20': 0.0518},
    'baby':        {'Recall@10': 0.0675, 'Recall@20': 0.1054, 'NDCG@10': 0.0365, 'NDCG@20': 0.0462},
    'electronics': {'Recall@10': 0.0460, 'Recall@20': 0.0680, 'NDCG@10': 0.0260, 'NDCG@20': 0.0317},
}

DATASET_PROFILES = {
    'sports': {
        'name':        'Amazon Sports',
        'domain':      'E-commerce (Visual + Textual)',
        'scale':       '35,598 Users | 18,357 Items | 296K Interactions',
        'color':       '#ff7f0e',
        'approx_mins': 42.0,
    },
    'baby': {
        'name':        'Amazon Baby',
        'domain':      'E-commerce (Visual + Textual)',
        'scale':       '19,445 Users | 7,050 Items | 160K Interactions',
        'color':       '#1f77b4',
        'approx_mins': 18.0,
    },
    'electronics': {
        'name':        'Amazon Electronics',
        'domain':      'E-commerce (Visual + Textual)',
        'scale':       '192,403 Users | 63,001 Items | 1.68M Interactions',
        'color':       '#2ca02c',
        'approx_mins': 160.0,
    },
}

vram_profile = {'sports': [], 'baby': [], 'electronics': []}
STAIR6_V1_RESULTS = {}

def vram_monitor(key, stop_event, interval=2.0):
    nvml_ok = False
    handle = None
    try:
        import pynvml
        pynvml.nvmlInit()
        handle = pynvml.nvmlDeviceGetHandleByIndex(0)
        nvml_ok = True
    except Exception:
        pass

    while not stop_event.is_set():
        val = 0.0
        if nvml_ok and handle:
            try:
                info = pynvml.nvmlDeviceGetMemoryInfo(handle)
                val = info.used / (1024 ** 2)
            except Exception:
                pass
        elif torch.cuda.is_available():
            try:
                val = torch.cuda.max_memory_allocated() / (1024 ** 2)
            except Exception:
                pass
        if val > 0:
            vram_profile[key].append(val)
        time.sleep(interval)

    if nvml_ok:
        try:
            import pynvml
            pynvml.nvmlShutdown()
        except Exception:
            pass

def extract_test_metrics(log_path, artifact_dir=None):
    if artifact_dir:
        sp_path = Path(artifact_dir) / 'selected_test_metrics.json'
        if sp_path.exists():
            try:
                with open(sp_path, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                return data.get('epoch', 0), data.get('test_metrics', {})
            except Exception:
                pass

    if not os.path.exists(log_path):
        return 0, {}

    best_epoch = 0
    metrics = {}
    with open(log_path, 'r', encoding='utf-8', errors='replace') as f:
        for line in f:
            if 'Best epoch:' in line:
                m = re.search(r'Best epoch:\s*(\d+)', line)
                if m:
                    best_epoch = int(m.group(1))
            if 'Test metrics' in line or 'Evaluating on TEST' in line:
                for k in TRACKED_METRICS:
                    p = rf"{re.escape(k)}[:\s=]+([0-9\.]+)"
                    m2 = re.search(p, line)
                    if m2:
                        metrics[k] = float(m2.group(1))
    return best_epoch, metrics

def extract_best_validation(log_path):
    if not os.path.exists(log_path):
        return 0, {}
    best_epoch = 0
    val_metrics = {}
    with open(log_path, 'r', encoding='utf-8', errors='replace') as f:
        for line in f:
            if 'Validation' in line or 'Evaluating on VALID' in line:
                m_ep = re.search(r'Epoch\s*(\d+)', line, re.IGNORECASE)
                ep = int(m_ep.group(1)) if m_ep else 0
                cur_m = {}
                for k in TRACKED_METRICS:
                    p = rf"{re.escape(k)}[:\s=]+([0-9\.]+)"
                    m = re.search(p, line)
                    if m:
                        cur_m[k] = float(m.group(1))
                if cur_m.get('NDCG@20', 0) > val_metrics.get('NDCG@20', 0):
                    val_metrics = cur_m
                    best_epoch = ep
    return best_epoch, val_metrics

def parse_telemetry_jsonl(artifact_dir):
    p = Path(artifact_dir) / 'training_telemetry.jsonl'
    if not p.exists():
        return []
    records = []
    with open(p, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    records.append(json.loads(line))
                except Exception:
                    pass
    return records

def parse_training_losses(log_path, telemetry=None):
    if telemetry:
        bpr = [(r['epoch'], r.get('bpr_loss', 0.0)) for r in telemetry if 'bpr_loss' in r]
        cl = [(r['epoch'], r.get('weighted_nlgcl_loss', r.get('nlgcl_loss', r.get('cl_loss', 0.0)))) for r in telemetry if any(k in r for k in ('weighted_nlgcl_loss', 'nlgcl_loss', 'cl_loss'))]
        total = [(r['epoch'], r.get('loss', 0.0)) for r in telemetry if 'loss' in r]
        if total:
            return bpr, cl, total

    if not os.path.exists(log_path):
        return [], [], []

    bpr, cl, total = [], [], []
    with open(log_path, 'r', encoding='utf-8', errors='replace') as f:
        for line in f:
            m_ep = re.search(r'Epoch\s*(\d+)', line)
            if not m_ep:
                continue
            ep = int(m_ep.group(1))
            m_tot = re.search(r'Loss[:\s=]+([0-9\.]+)', line, re.IGNORECASE)
            m_bpr = re.search(r'BPR[:\s=]+([0-9\.]+)', line, re.IGNORECASE)
            m_cl = re.search(r'NLGCL[:\s=]+([0-9\.]+)', line, re.IGNORECASE)
            if m_tot:
                total.append((ep, float(m_tot.group(1))))
            if m_bpr:
                bpr.append((ep, float(m_bpr.group(1))))
            if m_cl:
                cl.append((ep, float(m_cl.group(1))))
    return bpr, cl, total

def parse_valid_metric(log_path, metric='NDCG@20'):
    if not os.path.exists(log_path):
        return []
    pattern = rf'Epoch\s*(\d+).*?{re.escape(metric)}[:\s=]+([0-9\.]+)'
    matches = []
    with open(log_path, 'r', encoding='utf-8', errors='replace') as f:
        content = f.read()
        matches = re.findall(pattern, content, re.DOTALL | re.IGNORECASE)
    return [(int(ep), float(v)) for ep, v in matches]

def run_training_stair6_v1(key, config_yaml=None, data_root=None, log_path=None, artifact_dir=None, **kwargs):
    """Executes STAIR6-v1 training with real-time streaming, metadata preflight and fail-fast handling."""
    cfg_dict = STAIR6_V1_CONFIGS.get(key, {}).copy()
    cfg_dict.update(kwargs)

    config_yaml = config_yaml or cfg_dict.get('yaml')
    label = f"{key}_STAIR6-v1_{cfg_dict.get('category_mode', 'cae')}_seed{cfg_dict.get('seed', 1)}"
    log_path = log_path or str(Path(cfg_dict['log']).parent / (label + '.log'))
    artifact_dir = artifact_dir or str(Path(cfg_dict['artifact_dir']).parent / label)
    resuming = bool(cfg_dict.get('resume_from'))
    if not resuming and (Path(log_path).exists() or Path(artifact_dir).exists()):
        print(f"⚠️ Thông báo: Tái lập kịch bản run cho {label}.")
        if Path(artifact_dir).exists():
            for stale in ('manifest.json', 'training_checkpoint.pt', 'training_telemetry.jsonl', 'best_model.pt', 'last_model.pt', 'selected_test_metrics.json'):
                sp_file = Path(artifact_dir) / stale
                if sp_file.exists():
                    try:
                        sp_file.unlink()
                    except OSError:
                        pass
    STAIR6_V1_CONFIGS[key].update(log=log_path, artifact_dir=artifact_dir)

    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    os.makedirs(artifact_dir, exist_ok=True)

    # Tìm file runner main_stair6_v1.py
    _possible_runners = [
        '/kaggle/working/STAIR-Enhanced/main_stair6_v1.py',
        '/kaggle/working/main_stair6_v1.py',
        os.path.expanduser('~/STAIR-Enhanced/main_stair6_v1.py'),
        'main_stair6_v1.py',
    ]
    runner_py = next((r for r in _possible_runners if os.path.exists(r)), 'main_stair6_v1.py')

    # Tìm file config yaml
    if not os.path.exists(config_yaml):
        cand_y1 = os.path.join('configs', os.path.basename(config_yaml))
        cand_y2 = os.path.expanduser(f'~/STAIR-Enhanced/configs/{os.path.basename(config_yaml)}')
        cand_y3 = f'/kaggle/working/STAIR-Enhanced/configs/{os.path.basename(config_yaml)}'
        cand_y4 = f'/kaggle/working/configs/{os.path.basename(config_yaml)}'
        for cand in [cand_y1, cand_y2, cand_y3, cand_y4]:
            if os.path.exists(cand):
                config_yaml = cand
                break

    # Xác định data_root phù hợp
    if data_root is None:
        data_root = '/kaggle/data' if os.path.exists('/kaggle/data') else 'data'

    # Kiểm tra phương thức danh mục: latent (tự động phân cụm feature) vs metadata (ASIN mapping)
    _cat_mode = cfg_dict.get('category_mode', 'cae')
    _cat_source = cfg_dict.get('category_source', 'latent')
    if _cat_mode == 'cae' and _cat_source == 'metadata':
        _ds_folder = os.path.join(data_root, 'Processed', cfg_dict.get('dataset', ''))
        if not os.path.exists(_ds_folder):
            _ds_folder = os.path.join(data_root, cfg_dict.get('dataset', ''))
        _mapping_file = os.path.join(_ds_folder, 'item_id_to_asin.json')
        if not os.path.exists(_mapping_file):
            print(f"ℹ️ Lưu ý khoa học: Chưa phát hiện file metadata ASIN mapping ({_mapping_file}).")
            print(f"   ↳ Tự động chạy chế độ đối chứng cách ly (--category-mode off) theo chuẩn V8 backbone.")
            _cat_mode = 'off'
    elif _cat_mode == 'cae' and _cat_source == 'latent':
        print(f"✨ Kích hoạt cơ chế STAIR6-v1 LCAE-WMSG-CSE (Latent Category-Association Enrichment)")
        print(f"   ↳ Tự động gom cụm đặc trưng đa phương thức (Spherical K-Means) thành {cfg_dict.get('latent_clusters', 50)} cụm nội dung tiềm ẩn.")

    # Bảo vệ bổ sung: đảm bảo thư mục chunks không phải là symlink trỏ vào read-only filesystem
    for _base in ['/kaggle/data/Processed', '/kaggle/data']:
        _cand_ds = os.path.join(_base, cfg_dict.get('dataset', ''))
        if os.path.islink(_cand_ds):
            _tgt = os.path.realpath(_cand_ds)
            if '/kaggle/input' in _tgt or not os.access(_tgt, os.W_OK):
                try:
                    os.unlink(_cand_ds)
                    os.makedirs(_cand_ds, exist_ok=True)
                    for _it in os.listdir(_tgt):
                        if _it == 'chunks':
                            continue
                        _s_it = os.path.join(_tgt, _it)
                        _d_it = os.path.join(_cand_ds, _it)
                        try:
                            os.symlink(_s_it, _d_it)
                        except Exception:
                            shutil.copy2(_s_it, _d_it)
                except Exception:
                    pass
        _chunks_dir = os.path.join(_cand_ds, 'chunks')
        if os.path.islink(_chunks_dir):
            try:
                os.unlink(_chunks_dir)
            except Exception:
                pass
        try:
            os.makedirs(_chunks_dir, exist_ok=True)
        except Exception:
            pass

    print('=' * 80)
    print(f"🚀 KHỞI ĐỘNG TIẾN TRÌNH HUẤN LUYỆN STAIR6-v1 / CAE-WMSG-CSE: {key.upper()}")
    print(f"  * Dataset Key         : {key}")
    print(f"  * YAML Configuration  : {config_yaml}")
    print(f"  * Log Path            : {log_path}")
    print(f"  * Artifact Directory  : {artifact_dir}")
    print(f"  * Category Mode       : {_cat_mode}")
    print(f"  * Category Strength   : {cfg_dict.get('category_strength', 0.5)}")
    print(f"  * Category Shrinkage  : {cfg_dict.get('category_shrinkage', 5.0)}")
    print(f"  * WMSG Edge Power (α) : {cfg_dict.get('edge_power', 2.0)}")
    print(f"  * WMSG Edge Floor     : {cfg_dict.get('edge_floor', 0.05)}")
    print(f"  * CSE Operator (η)    : {cfg_dict.get('eta', 0.1)}")
    print(f"  * Seed                : {cfg_dict.get('seed', 1)}")
    print('=' * 80)

    stop_evt = threading.Event()
    th = threading.Thread(target=vram_monitor, args=(key, stop_evt), daemon=True)
    th.start()

    t0 = time.time()
    device_str = str(cfg_dict.get('device', '0' if torch.cuda.is_available() else 'cpu'))
    cmd = [
        sys.executable, '-u', str(runner_py),
        '--config', str(config_yaml),
        '--root', str(data_root),
        '--device', str(device_str),
        '--ranking', 'full',
        '--artifact-dir', str(artifact_dir),
        '--category-mode', str(_cat_mode),
    ]

    defaults = {
        'v8_arm': 'WMSG-core', 'semantic_mode': 'weighted', 'edge_power': 2.0, 'edge_floor': 0.05,
        'semantic_mix': 1.0, 'alignment': 'off', 'relation_gate': 'off', 'relation_strength': 0.0,
        'bsc_residual': 0.0, 'lambda_dirichlet': 0.0, 'seed': 1, 'eval_freq': 5, 'eta': 0.1,
        'k_cf': 5, 'c_min': 2, 't_shrinkage': 5.0, 'cf_block_size': 64, 'cf_memory_budget_mib': 128,
        'lambda_nlgcl': 0.01, 'nlgcl_tau': 0.2, 'nlgcl_G': 1, 'nlgcl_alpha': 0.5, 'cl_chunk_size': 1024,
        'knn_chunk_size': 1024, 'knn_device': 'cpu',
        'category_strength': 0.5, 'category_min_count': 2, 'category_shrinkage': 5.0,
        'category_source': 'latent', 'category_memory_budget_mib': 128.0, 'category_probe_dim': 8, 'category_probe_seed': 2718,
        'latent_clusters': 50, 'latent_projection_dim': 64, 'latent_iterations': 20, 'latent_chunk_size': 2048, 'latent_seed': 2718, 'latent_memory_budget_mib': 128.0,
        'graph_cache_dir': '/kaggle/working/runs/stair6_v1/graph_cache',
    }
    settings = {**defaults, **cfg_dict}
    for k_item in defaults:
        if k_item != 'category_mode':
            cmd += ['--' + k_item.replace('_', '-'), str(settings[k_item])]
    for k_item in ('epochs', 'batch_size', 'gamma', 'lr', 'weight_decay', 'num_workers'):
        if k_item in cfg_dict:
            cmd += ['--' + k_item.replace('_', '-'), str(cfg_dict[k_item])]
    if cfg_dict.get('resume_from'):
        cmd += ['--resume-from', str(cfg_dict['resume_from'])]

    print('Command:', ' '.join(cmd))
    env = os.environ.copy()
    _active_dir = os.path.dirname(os.path.abspath(runner_py))
    env['PYTHONPATH'] = os.pathsep.join([p for p in ['.', _active_dir, '/kaggle/working', '/kaggle/working/STAIR-Enhanced', os.environ.get('PYTHONPATH', '')] if p])
    env['PYTHONUTF8'] = '1'

    try:
        with open(log_path, 'a' if resuming else 'w', encoding='utf-8') as lf:
            proc = subprocess.Popen(
                cmd,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                encoding='utf-8', errors='replace',
                bufsize=1,
                env=env,
                cwd=_active_dir,
            )
            for line in proc.stdout:
                sys.stdout.write(line)
                sys.stdout.flush()
                lf.write(line)
                lf.flush()
            proc.wait()
            ret_code = proc.returncode
    finally:
        stop_evt.set()
        th.join(timeout=3.0)

    elapsed = time.time() - t0
    print(f"⏱️ Thời gian thực thi ({key.upper()}): {elapsed/60:.2f} phút | Trạng thái exit code: {ret_code}")
    if ret_code != 0:
        raise RuntimeError(f"Tiến trình huấn luyện {key.upper()} thất bại với mã lỗi {ret_code}. Kiểm tra log: {log_path}")
    return elapsed


## Cell 5 📋 Cấu hình Siêu tham số STAIR6-v1 CAE-WMSG-CSE (Dataset-Adaptive Matrix)
- Ma trận siêu tham số thích ứng theo từng tập dữ liệu (`Amazon Sports`, `Amazon Baby`, `Amazon Electronics`)
- Tự động nhận diện GPU dung lượng lớn (**NVIDIA Blackwell / RTX PRO 6000 96GB**) để tối ưu `cl_chunk_size`
- Tích hợp Ledger bộ nhớ Tensor lý thuyết ($d=64$, trần VRAM dưới 240 MiB cho Sports, 220 MiB cho Baby, và cổng an toàn 800 MiB cho Electronics)


In [ ]:
# Cell 5: Cấu hình Siêu tham số STAIR6-v1 CAE-WMSG-CSE (Dataset-Adaptive Matrix)
import os, uuid, torch
from datetime import datetime
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:6]

for _d in ['/kaggle/working/logs/stair6_v1', '/kaggle/working/reports', '/kaggle/working/runs/stair6_v1/graph_cache',
           os.path.expanduser('~/logs/stair6_v1'), os.path.expanduser('~/reports')]:
    try:
        os.makedirs(_d, exist_ok=True)
    except Exception:
        pass

# Nhận diện GPU VRAM: Nếu > 32GB VRAM (Blackwell RTX PRO 6000 96GB), tự động mở rộng chunk size để đạt hiệu năng tối đa
_is_blackwell = torch.cuda.is_available() and torch.cuda.get_device_properties(0).total_memory > 32 * (1024**3)
_elec_cl_chunk = 1024 if _is_blackwell else 128

if _is_blackwell:
    print(f"🚀 PHÁT HIỆN GPU DUNG LƯỢNG LỚN: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB VRAM).")
    print(f"  ↳ Tối ưu Electronics: cl_chunk_size={_elec_cl_chunk} (Tăng tốc tối đa).")

STAIR6_V1_CONFIGS = {
    'sports': {
        'dataset':            'Amazon2014Sports_550_MMRec',
        'yaml':               '/kaggle/working/STAIR-Enhanced/configs/Amazon2014Sports_STAIR6_v1.yaml',
        'log':                f'/kaggle/working/logs/stair6_v1/{RUN_ID}/sports_stair6_v1.log',
        'artifact_dir':       f'/kaggle/working/runs/stair6_v1/{RUN_ID}/sports_CAE-WMSG_seed1',
        'category_mode':      'cae',
        'category_source':    'latent',
        'latent_clusters':    50,
        'latent_iterations':  20,
        'latent_chunk_size':  2048,
        'latent_seed':        2718,
        'category_strength':  0.5,
        'category_shrinkage': 5.0,
        'v8_arm':             'WMSG-core',
        'semantic_mode':      'weighted',
        'edge_power':         2.0,
        'edge_floor':         0.05,
        'semantic_mix':       1.0,
        'alignment':          'off',
        'relation_gate':      'off',
        'eta':                0.1,
        'k_cf':               5,
        'c_min':              2,
        't_shrinkage':        5.0,
        'cf_block_size':      64,
        'lambda_nlgcl':       0.01,
        'nlgcl_tau':          0.2,
        'nlgcl_G':            1,
        'nlgcl_alpha':        0.5,
        'cl_chunk_size':      1024,
        'seed':               1,
        'eval_freq':          5,
    },
    'baby': {
        'dataset':            'Amazon2014Baby_550_MMRec',
        'yaml':               '/kaggle/working/STAIR-Enhanced/configs/Amazon2014Baby_STAIR6_v1.yaml',
        'log':                f'/kaggle/working/logs/stair6_v1/{RUN_ID}/baby_stair6_v1.log',
        'artifact_dir':       f'/kaggle/working/runs/stair6_v1/{RUN_ID}/baby_CAE-WMSG_seed1',
        'category_mode':      'cae',
        'category_source':    'latent',
        'latent_clusters':    50,
        'latent_iterations':  20,
        'latent_chunk_size':  2048,
        'latent_seed':        2718,
        'category_strength':  0.5,
        'category_shrinkage': 5.0,
        'v8_arm':             'WMSG-core',
        'semantic_mode':      'weighted',
        'edge_power':         2.0,
        'edge_floor':         0.05,
        'semantic_mix':       1.0,
        'alignment':          'off',
        'relation_gate':      'off',
        'eta':                0.1,
        'k_cf':               5,
        'c_min':              2,
        't_shrinkage':        5.0,
        'cf_block_size':      64,
        'lambda_nlgcl':       0.01,
        'nlgcl_tau':          0.2,
        'nlgcl_G':            1,
        'nlgcl_alpha':        0.5,
        'cl_chunk_size':      1024,
        'seed':               1,
        'eval_freq':          5,
    },
    'electronics': {
        'dataset':            'Amazon2014Electronics_550_MMRec',
        'yaml':               '/kaggle/working/STAIR-Enhanced/configs/Amazon2014Electronics_STAIR6_v1.yaml',
        'log':                f'/kaggle/working/logs/stair6_v1/{RUN_ID}/electronics_stair6_v1.log',
        'artifact_dir':       f'/kaggle/working/runs/stair6_v1/{RUN_ID}/electronics_CAE-WMSG_seed1',
        'category_mode':      'cae',
        'category_source':    'latent',
        'latent_clusters':    50,
        'latent_iterations':  20,
        'latent_chunk_size':  2048,
        'latent_seed':        2718,
        'category_strength':  0.5,
        'category_shrinkage': 5.0,
        'v8_arm':             'WMSG-core',
        'semantic_mode':      'weighted',
        'edge_power':         2.0,
        'edge_floor':         0.05,
        'semantic_mix':       1.0,
        'alignment':          'off',
        'relation_gate':      'off',
        'eta':                0.1,
        'k_cf':               5,
        'c_min':              2,
        't_shrinkage':        5.0,
        'cf_block_size':      64,
        'lambda_nlgcl':       0.01,
        'nlgcl_tau':          0.2,
        'nlgcl_G':            1,
        'nlgcl_alpha':        0.5,
        'cl_chunk_size':      _elec_cl_chunk,
        'seed':               1,
        'eval_freq':          5,
    },
}

print('✅ Ma trận cấu hình STAIR6-v1 CAE-WMSG-CSE đã nạp thành công:')
for _k, _v in STAIR6_V1_CONFIGS.items():
    print(f"  * {_k.upper():12s} -> Mode: {_v['category_mode']} | Source: {_v.get('category_source', 'latent')} ({_v.get('latent_clusters', 50)} clusters) | Shrinkage: {_v['category_shrinkage']}")


## Cell 6a 🏋️ Huấn luyện Pha A — Amazon Sports (Pilot STAIR6-v1 / CAE-WMSG-CSE)
- Khởi động huấn luyện STAIR6-v1 trên tập dữ liệu Amazon Sports (~296K tương tác, 500 epochs)
- Tự động trích xuất điểm Validation NDCG@20 tốt nhất và nạp checkpoint để đánh giá Full-Ranking Test


In [ ]:
# Cell 6a: Training STAIR6-v1 on Amazon Sports (Pha A: Pilot)
sports_time = run_training_stair6_v1('sports')
ep_sports, metrics_sports = extract_test_metrics(STAIR6_V1_CONFIGS['sports']['log'], STAIR6_V1_CONFIGS['sports']['artifact_dir'])
val_ep_sports, val_m_sports = extract_best_validation(STAIR6_V1_CONFIGS['sports']['log'])
STAIR6_V1_RESULTS['sports'] = {
    'metrics': metrics_sports,
    'best_epoch': ep_sports,
    'val_metrics': val_m_sports,
    'train_time': sports_time,
}

print("=" * 80)
print(f"🏆 KẾT QUẢ NGHIỆM THU AMAZON SPORTS (STAIR6-v1 / Best Epoch: {ep_sports}):")
for m, v in metrics_sports.items():
    v8_val = V8_64_REF['sports'].get(m, 0.0)
    delta_v8 = ((v - v8_val) / v8_val) * 100 if v8_val else 0.0
    print(f"  * {m:12s}: {v:.4f} (v8-64D: {v8_val:.4f} | Δ vs v8: {delta_v8:+.2f}%)")
print("=" * 80)


## Cell 6b ⚡ Biểu đồ Tiêu thụ Bộ nhớ Tensor Mô hình — Amazon Sports
- Vẽ đồ thị phân tích chi tiết mức tiêu thụ VRAM với đường trần lý thuyết (< 240 MiB Flat Memory Budget)
- Minh chứng tính bảo toàn chi phí phần cứng của CAE: zero overhead trên GPU


In [ ]:
# Cell 6b: Model Tensor VRAM Profile — Amazon Sports (Paper Standard)
_sports_art = STAIR6_V1_CONFIGS['sports']['artifact_dir']
_telemetry_sports = parse_telemetry_jsonl(_sports_art)

plt.figure(figsize=(9, 4.5), dpi=150)
if _telemetry_sports:
    _eps = [r['epoch'] for r in _telemetry_sports]
    _vram_mb = [r.get('peak_allocated_gib', 0) * 1024 if 'peak_allocated_gib' in r else r.get('max_memory_allocated_bytes', 0) / (1024**2) for r in _telemetry_sports]
    plt.plot(_eps, _vram_mb, color='#ff7f0e', lw=2.2, marker='o', ms=4, label='Peak Allocated VRAM (MB)')
    plt.axhline(240, color='red', ls=':', lw=1.5, label='Theoretical Ceiling Budget (240 MiB)')
    plt.title('Amazon Sports — Peak Model Tensor VRAM Profile (STAIR6-v1 LCAE-WMSG-CSE)', fontsize=12, fontweight='bold')
    plt.xlabel('Epoch', fontsize=10)
    plt.ylabel('Allocated VRAM (MB)', fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend(frameon=True)
elif 'sports' in vram_profile and len(vram_profile['sports']) > 0:
    plt.plot(vram_profile['sports'], color='#ff7f0e', lw=1.8, label='Host NVML GPU Used (MB)')
    plt.title('Amazon Sports — Host GPU Memory Profile (NVML)', fontsize=12, fontweight='bold')
    plt.xlabel('Polling Step (~2s)', fontsize=10)
    plt.ylabel('Memory (MB)', fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend(frameon=True)
else:
    print("Chưa có dữ liệu VRAM telemetry cho Amazon Sports.")
plt.tight_layout()
plt.savefig('/kaggle/working/reports/vram_profile_sports.png', dpi=150)
plt.show()


## Cell 6c 📈 Động Lực Học & Quá Trình Hội Tụ — Amazon Sports
- Trực quan hóa hệ thống hàm mất mát (Total Loss, BPR Loss) song song với chất lượng xếp hạng Validation NDCG@20 trên trục kép


In [ ]:
# Cell 6c: Learning Dynamics & Convergence Profiles — Amazon Sports
_fig, _ax1 = plt.subplots(figsize=(10, 4.8), dpi=150)

_sports_art = STAIR6_V1_CONFIGS['sports']['artifact_dir']
_telemetry_sports = parse_telemetry_jsonl(_sports_art)
_log_sports = STAIR6_V1_CONFIGS['sports']['log']
_bpr_l, _cl_l, _tot_l = parse_training_losses(_log_sports, _telemetry_sports)
_val_ndcg = parse_valid_metric(_log_sports, 'NDCG@20')

if _tot_l:
    _x_tot, _y_tot = zip(*_tot_l)
    _ax1.plot(_x_tot, _y_tot, color='#1f77b4', lw=2.0, label='Total Loss (BPR + λ·NLGCL)')
if _bpr_l:
    _x_bpr, _y_bpr = zip(*_bpr_l)
    _ax1.plot(_x_bpr, _y_bpr, color='#1f77b4', lw=1.5, ls='--', alpha=0.7, label='BPR Loss')

_ax1.set_xlabel('Epoch', fontsize=10)
_ax1.set_ylabel('Training Loss', color='#1f77b4', fontsize=10)
_ax1.tick_params(axis='y', labelcolor='#1f77b4')
_ax1.grid(True, linestyle='--', alpha=0.4)

if _val_ndcg:
    _ax2 = _ax1.twinx()
    _x_v, _y_v = zip(*_val_ndcg)
    _ax2.plot(_x_v, _y_v, color='#d62728', lw=2.0, marker='s', ms=4, label='Validation NDCG@20')
    _ax2.set_ylabel('Validation NDCG@20', color='#d62728', fontsize=10)
    _ax2.tick_params(axis='y', labelcolor='#d62728')

plt.title('Amazon Sports — Learning Dynamics & Convergence Profiles (STAIR6-v1)', fontsize=12, fontweight='bold')
_fig.tight_layout()
plt.savefig('/kaggle/working/reports/learning_curve_sports.png', dpi=150)
plt.show()


## Cell 7a 🏋️ Huấn luyện Pha B — Amazon Baby (Kiểm chứng An toàn STAIR6-v1 / CAE-WMSG-CSE)
- Huấn luyện mô hình trên tập dữ liệu Amazon Baby (~160K tương tác) kiểm chứng tính ổn định của trọng số WMSG và CAE
- Đảm bảo tính nhất quán giữa kiểm tra metadata và huấn luyện thực tế


In [ ]:
# Cell 7a: Training STAIR6-v1 on Amazon Baby (Pha B: Confirmatory)
baby_time = run_training_stair6_v1('baby')
ep_baby, metrics_baby = extract_test_metrics(STAIR6_V1_CONFIGS['baby']['log'], STAIR6_V1_CONFIGS['baby']['artifact_dir'])
val_ep_baby, val_m_baby = extract_best_validation(STAIR6_V1_CONFIGS['baby']['log'])
STAIR6_V1_RESULTS['baby'] = {
    'metrics': metrics_baby,
    'best_epoch': ep_baby,
    'val_metrics': val_m_baby,
    'train_time': baby_time,
}

print("=" * 80)
print(f"🏆 KẾT QUẢ NGHIỆM THU AMAZON BABY (STAIR6-v1 / Best Epoch: {ep_baby}):")
for m, v in metrics_baby.items():
    v8_val = V8_64_REF['baby'].get(m, 0.0)
    delta_v8 = ((v - v8_val) / v8_val) * 100 if v8_val else 0.0
    print(f"  * {m:12s}: {v:.4f} (v8-64D: {v8_val:.4f} | Δ vs v8: {delta_v8:+.2f}%)")
print("=" * 80)


## Cell 7b ⚡ Biểu đồ Tiêu thụ Bộ nhớ Tensor Mô hình — Amazon Baby
- Phân tích chi tiết mức tiêu thụ VRAM thực tế đối chiếu đường trần an toàn lý thuyết (< 220 MiB Flat Memory Budget)


In [ ]:
# Cell 7b: Model Tensor VRAM Profile — Amazon Baby (Paper Standard)
_baby_art = STAIR6_V1_CONFIGS['baby']['artifact_dir']
_telemetry_baby = parse_telemetry_jsonl(_baby_art)

plt.figure(figsize=(9, 4.5), dpi=150)
if _telemetry_baby:
    _eps = [r['epoch'] for r in _telemetry_baby]
    _vram_mb = [r.get('peak_allocated_gib', 0) * 1024 if 'peak_allocated_gib' in r else r.get('max_memory_allocated_bytes', 0) / (1024**2) for r in _telemetry_baby]
    plt.plot(_eps, _vram_mb, color='#1f77b4', lw=2.2, marker='o', ms=4, label='Peak Allocated VRAM (MB)')
    plt.axhline(220, color='red', ls=':', lw=1.5, label='Theoretical Ceiling Budget (220 MiB)')
    plt.title('Amazon Baby — Peak Model Tensor VRAM Profile (STAIR6-v1 LCAE-WMSG-CSE)', fontsize=12, fontweight='bold')
    plt.xlabel('Epoch', fontsize=10)
    plt.ylabel('Allocated VRAM (MB)', fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend(frameon=True)
elif 'baby' in vram_profile and len(vram_profile['baby']) > 0:
    plt.plot(vram_profile['baby'], color='#1f77b4', lw=1.8, label='Host NVML GPU Used (MB)')
    plt.title('Amazon Baby — Host GPU Memory Profile (NVML)', fontsize=12, fontweight='bold')
    plt.xlabel('Polling Step (~2s)', fontsize=10)
    plt.ylabel('Memory (MB)', fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend(frameon=True)
else:
    print("Chưa có dữ liệu VRAM telemetry cho Amazon Baby.")
plt.tight_layout()
plt.savefig('/kaggle/working/reports/vram_profile_baby.png', dpi=150)
plt.show()


## Cell 7c 📈 Động Lực Học & Quá Trình Hội Tụ — Amazon Baby
- Trực quan hóa đường cong hàm mất mát huấn luyện và chất lượng đánh giá Validation NDCG@20


In [ ]:
# Cell 7c: Learning Dynamics & Convergence Profiles — Amazon Baby
_fig, _ax1 = plt.subplots(figsize=(10, 4.8), dpi=150)

_baby_art = STAIR6_V1_CONFIGS['baby']['artifact_dir']
_telemetry_baby = parse_telemetry_jsonl(_baby_art)
_log_baby = STAIR6_V1_CONFIGS['baby']['log']
_bpr_l, _cl_l, _tot_l = parse_training_losses(_log_baby, _telemetry_baby)
_val_ndcg = parse_valid_metric(_log_baby, 'NDCG@20')

if _tot_l:
    _x_tot, _y_tot = zip(*_tot_l)
    _ax1.plot(_x_tot, _y_tot, color='#1f77b4', lw=2.0, label='Total Loss (BPR + λ·NLGCL)')
if _bpr_l:
    _x_bpr, _y_bpr = zip(*_bpr_l)
    _ax1.plot(_x_bpr, _y_bpr, color='#1f77b4', lw=1.5, ls='--', alpha=0.7, label='BPR Loss')

_ax1.set_xlabel('Epoch', fontsize=10)
_ax1.set_ylabel('Training Loss', color='#1f77b4', fontsize=10)
_ax1.tick_params(axis='y', labelcolor='#1f77b4')
_ax1.grid(True, linestyle='--', alpha=0.4)

if _val_ndcg:
    _ax2 = _ax1.twinx()
    _x_v, _y_v = zip(*_val_ndcg)
    _ax2.plot(_x_v, _y_v, color='#d62728', lw=2.0, marker='s', ms=4, label='Validation NDCG@20')
    _ax2.set_ylabel('Validation NDCG@20', color='#d62728', fontsize=10)
    _ax2.tick_params(axis='y', labelcolor='#d62728')

plt.title('Amazon Baby — Learning Dynamics & Convergence Profiles (STAIR6-v1)', fontsize=12, fontweight='bold')
_fig.tight_layout()
plt.savefig('/kaggle/working/reports/learning_curve_baby.png', dpi=150)
plt.show()


## Cell 8a 🏋️ Huấn luyện Pha C — Amazon Electronics (~1.7M Tương tác, Scale-up STAIR6-v1)
- Huấn luyện mô hình quy mô lớn trên Amazon Electronics (~1.68 triệu tương tác, 192K Users, 63K Items)
- Kiểm chứng khả năng kháng OOM nhờ phân mảnh khối `cf_block_size=64`, `cl_chunk_size=128` (hoặc 1024 trên Blackwell)


In [ ]:
# Cell 8a: Training STAIR6-v1 on Amazon Electronics (Pha C: Scale-up)
electronics_time = run_training_stair6_v1('electronics')
ep_elec, metrics_elec = extract_test_metrics(STAIR6_V1_CONFIGS['electronics']['log'], STAIR6_V1_CONFIGS['electronics']['artifact_dir'])
val_ep_elec, val_m_elec = extract_best_validation(STAIR6_V1_CONFIGS['electronics']['log'])
STAIR6_V1_RESULTS['electronics'] = {
    'metrics': metrics_elec,
    'best_epoch': ep_elec,
    'val_metrics': val_m_elec,
    'train_time': electronics_time,
}

print("=" * 80)
print(f"🏆 KẾT QUẢ NGHIỆM THU AMAZON ELECTRONICS (STAIR6-v1 / Best Epoch: {ep_elec}):")
for m, v in metrics_elec.items():
    v8_val = V8_64_REF['electronics'].get(m, 0.0)
    delta_v8 = ((v - v8_val) / v8_val) * 100 if v8_val else 0.0
    print(f"  * {m:12s}: {v:.4f} (v8-64D: {v8_val:.4f} | Δ vs v8: {delta_v8:+.2f}%)")
print("=" * 80)


## Cell 8b ⚡ Biểu đồ Tiêu thụ Bộ nhớ Tensor Mô hình — Amazon Electronics (Cổng An toàn Phần cứng)
- Phân tích chi tiết mức tiêu thụ VRAM đối chiếu với Cổng An toàn Phần cứng Nghiêm ngặt (< 800 MiB Strict Hardware Gate)


In [ ]:
# Cell 8b: Model Tensor VRAM Profile — Amazon Electronics (Memory Gate Verification)
_elec_art = STAIR6_V1_CONFIGS['electronics']['artifact_dir']
_telemetry_elec = parse_telemetry_jsonl(_elec_art)

plt.figure(figsize=(9, 4.5), dpi=150)
if _telemetry_elec:
    _eps = [r['epoch'] for r in _telemetry_elec]
    _vram_mb = [r.get('peak_allocated_gib', 0) * 1024 if 'peak_allocated_gib' in r else r.get('max_memory_allocated_bytes', 0) / (1024**2) for r in _telemetry_elec]
    plt.plot(_eps, _vram_mb, color='#2ca02c', lw=2.2, marker='o', ms=4, label='Peak Allocated VRAM (MB)')
    plt.axhline(800, color='red', ls=':', lw=1.5, label='Strict Hardware Gate (800 MiB)')
    plt.title('Amazon Electronics — Peak Model Tensor VRAM Profile (STAIR6-v1)', fontsize=12, fontweight='bold')
    plt.xlabel('Epoch', fontsize=10)
    plt.ylabel('Allocated VRAM (MB)', fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend(frameon=True)
elif 'electronics' in vram_profile and len(vram_profile['electronics']) > 0:
    plt.plot(vram_profile['electronics'], color='#2ca02c', lw=1.8, label='Host NVML GPU Used (MB)')
    plt.title('Amazon Electronics — Host GPU Memory Profile (NVML)', fontsize=12, fontweight='bold')
    plt.xlabel('Polling Step (~2s)', fontsize=10)
    plt.ylabel('Memory (MB)', fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend(frameon=True)
else:
    print("Chưa có dữ liệu VRAM telemetry cho Amazon Electronics.")
plt.tight_layout()
plt.savefig('/kaggle/working/reports/vram_profile_electronics.png', dpi=150)
plt.show()


## Cell 8c 📈 Động Lực Học & Quá Trình Hội Tụ — Amazon Electronics
- Trực quan hóa động lực học hàm mất mát và chất lượng xếp hạng Validation NDCG@20 trên tập dữ liệu quy mô lớn


In [ ]:
# Cell 8c: Learning Dynamics & Convergence Profiles — Amazon Electronics
_fig, _ax1 = plt.subplots(figsize=(10, 4.8), dpi=150)

_elec_art = STAIR6_V1_CONFIGS['electronics']['artifact_dir']
_telemetry_elec = parse_telemetry_jsonl(_elec_art)
_log_elec = STAIR6_V1_CONFIGS['electronics']['log']
_bpr_l, _cl_l, _tot_l = parse_training_losses(_log_elec, _telemetry_elec)
_val_ndcg = parse_valid_metric(_log_elec, 'NDCG@20')

if _tot_l:
    _x_tot, _y_tot = zip(*_tot_l)
    _ax1.plot(_x_tot, _y_tot, color='#1f77b4', lw=2.0, label='Total Loss (BPR + λ·NLGCL)')
if _bpr_l:
    _x_bpr, _y_bpr = zip(*_bpr_l)
    _ax1.plot(_x_bpr, _y_bpr, color='#1f77b4', lw=1.5, ls='--', alpha=0.7, label='BPR Loss')

_ax1.set_xlabel('Epoch', fontsize=10)
_ax1.set_ylabel('Training Loss', color='#1f77b4', fontsize=10)
_ax1.tick_params(axis='y', labelcolor='#1f77b4')
_ax1.grid(True, linestyle='--', alpha=0.4)

if _val_ndcg:
    _ax2 = _ax1.twinx()
    _x_v, _y_v = zip(*_val_ndcg)
    _ax2.plot(_x_v, _y_v, color='#d62728', lw=2.0, marker='s', ms=4, label='Validation NDCG@20')
    _ax2.set_ylabel('Validation NDCG@20', color='#d62728', fontsize=10)
    _ax2.tick_params(axis='y', labelcolor='#d62728')

plt.title('Amazon Electronics — Learning Dynamics & Convergence Profiles (STAIR6-v1)', fontsize=12, fontweight='bold')
_fig.tight_layout()
plt.savefig('/kaggle/working/reports/learning_curve_electronics.png', dpi=150)
plt.show()


## Cell 9 📊 Bảng Đối chuẩn Đa thế hệ Toàn diện (Baseline vs v4 vs v6 vs v7 vs v8 vs STAIR6-v1)
- Bảng tổng hợp đối sánh hiệu năng xếp hạng trên cả 3 tập dữ liệu giữa tất cả các thế hệ STAIR
- Tính toán chính xác độ chênh lệch phần trăm ($\Delta$ vs Baseline, $\Delta$ vs v4, $\Delta$ vs v8-64D)
- Xuất file báo cáo cấu trúc `reports/stair6_v1_benchmark_summary.csv` sẵn sàng cho khóa luận tốt nghiệp


In [ ]:
# Cell 9: Bảng Đối chuẩn Đa thế hệ Toàn diện (STAIR Baseline vs v4 vs v6 vs v7 vs v8 vs STAIR6-v1)
import pandas as pd
from prettytable import PrettyTable

_table = PrettyTable()
_table.field_names = ['Dataset', 'Kiến trúc / Phiên bản', *TRACKED_METRICS, 'Δ vs Baseline (%)', 'Δ vs v4 (%)', 'Δ vs v8-64D (%)']

_df_rows = []

for _ds in ('sports', 'baby', 'electronics'):
    _ref_bl = BASELINE_REF[_ds]['NDCG@20']
    _ref_v4 = V4_REF[_ds]['NDCG@20']
    _ref_v6 = V6_REF[_ds]['NDCG@20']
    _ref_v7 = V7_REF[_ds]['NDCG@20']
    _ref_v8 = V8_64_REF[_ds]['NDCG@20']

    # 1. Baseline
    _table.add_row([_ds, 'STAIR (baseline)', *[f"{BASELINE_REF[_ds][_m]:.4f}" for _m in TRACKED_METRICS], '-', '-', '-'])
    _df_rows.append({'Dataset': _ds, 'Model': 'STAIR (baseline)', **{_m: BASELINE_REF[_ds][_m] for _m in TRACKED_METRICS}, 'Δ vs Baseline': 0.0, 'Δ vs v4': 0.0, 'Δ vs v8': 0.0})

    # 2. v4
    _d_v4_bl = 100 * (_ref_v4 - _ref_bl) / _ref_bl
    _table.add_row([_ds, 'STAIR5-v4 (NLGCL-CSE)', *[f"{V4_REF[_ds][_m]:.4f}" for _m in TRACKED_METRICS], f"{_d_v4_bl:+.2f}%", '-', '-'])
    _df_rows.append({'Dataset': _ds, 'Model': 'STAIR5-v4', **{_m: V4_REF[_ds][_m] for _m in TRACKED_METRICS}, 'Δ vs Baseline': _d_v4_bl, 'Δ vs v4': 0.0, 'Δ vs v8': 0.0})

    # 3. v6
    _d_v6_bl = 100 * (_ref_v6 - _ref_bl) / _ref_bl
    _d_v6_v4 = 100 * (_ref_v6 - _ref_v4) / _ref_v4
    _table.add_row([_ds, 'STAIR5-v6 (UCR)', *[f"{V6_REF[_ds][_m]:.4f}" for _m in TRACKED_METRICS], f"{_d_v6_bl:+.2f}%", f"{_d_v6_v4:+.2f}%", '-'])
    _df_rows.append({'Dataset': _ds, 'Model': 'STAIR5-v6', **{_m: V6_REF[_ds][_m] for _m in TRACKED_METRICS}, 'Δ vs Baseline': _d_v6_bl, 'Δ vs v4': _d_v6_v4, 'Δ vs v8': 0.0})

    # 4. v7
    _d_v7_bl = 100 * (_ref_v7 - _ref_bl) / _ref_bl
    _d_v7_v4 = 100 * (_ref_v7 - _ref_v4) / _ref_v4
    _table.add_row([_ds, 'STAIR5-v7 (UCR-D)', *[f"{V7_REF[_ds][_m]:.4f}" for _m in TRACKED_METRICS], f"{_d_v7_bl:+.2f}%", f"{_d_v7_v4:+.2f}%", '-'])
    _df_rows.append({'Dataset': _ds, 'Model': 'STAIR5-v7', **{_m: V7_REF[_ds][_m] for _m in TRACKED_METRICS}, 'Δ vs Baseline': _d_v7_bl, 'Δ vs v4': _d_v7_v4, 'Δ vs v8': 0.0})

    # 5. v8-64D
    _d_v8_bl = 100 * (_ref_v8 - _ref_bl) / _ref_bl
    _d_v8_v4 = 100 * (_ref_v8 - _ref_v4) / _ref_v4
    _table.add_row([_ds, 'STAIR5-v8 (WMSG-CSE)', *[f"{V8_64_REF[_ds][_m]:.4f}" for _m in TRACKED_METRICS], f"{_d_v8_bl:+.2f}%", f"{_d_v8_v4:+.2f}%", '-'])
    _df_rows.append({'Dataset': _ds, 'Model': 'STAIR5-v8 (WMSG-CSE)', **{_m: V8_64_REF[_ds][_m] for _m in TRACKED_METRICS}, 'Δ vs Baseline': _d_v8_bl, 'Δ vs v4': _d_v8_v4, 'Δ vs v8': 0.0})

    # 6. STAIR6-v1
    if _ds in STAIR6_V1_RESULTS and STAIR6_V1_RESULTS[_ds].get('metrics'):
        _s6_m = STAIR6_V1_RESULTS[_ds]['metrics']
        _ndcg_s6 = _s6_m.get('NDCG@20', 0.0)
        _d_s6_bl = 100 * (_ndcg_s6 - _ref_bl) / _ref_bl
        _d_s6_v4 = 100 * (_ndcg_s6 - _ref_v4) / _ref_v4
        _d_s6_v8 = 100 * (_ndcg_s6 - _ref_v8) / _ref_v8
        _table.add_row([
            _ds, 'STAIR6-v1 (LCAE-WMSG-CSE)',
            *[f"{_s6_m.get(_m, 0.0):.4f}" for _m in TRACKED_METRICS],
            f"{_d_s6_bl:+.2f}%", f"{_d_s6_v4:+.2f}%", f"{_d_s6_v8:+.2f}%"
        ])
        _df_rows.append({'Dataset': _ds, 'Model': 'STAIR6-v1 (LCAE-WMSG-CSE)', **{_m: _s6_m.get(_m, 0.0) for _m in TRACKED_METRICS}, 'Δ vs Baseline': _d_s6_bl, 'Δ vs v4': _d_s6_v4, 'Δ vs v8': _d_s6_v8})

print('=' * 110)
print('📊 BẢNG TỔNG HỢP SO SÁNH HIỆU NĂNG ĐA THẾ HỆ (STAIR Baseline vs v4 vs v6 vs v7 vs v8 vs STAIR6-v1):')
print('=' * 110)
print(_table)

# Xuất ra file CSV
_df_summary = pd.DataFrame(_df_rows)
os.makedirs('/kaggle/working/reports', exist_ok=True)
_summary_csv = '/kaggle/working/reports/stair6_v1_benchmark_summary.csv'
_df_summary.to_csv(_summary_csv, index=False)
print(f"\n✅ Báo cáo tổng hợp đã lưu tại: {_summary_csv}")


## Cell 10 📦 Đóng gói Lưu trữ Toàn bộ Minh chứng, Logs, Manifests & Checkpoints
- Nén toàn bộ Artifacts của STAIR6-v1 thành định dạng `.tar.gz` và `.zip`
- Đặt tại `/kaggle/working/stair6_v1_artifacts.tar.gz` sẵn sàng để tải về trực tiếp từ giao diện Kaggle Output


In [ ]:
# Cell 10: Đóng gói Lưu trữ Toàn bộ Minh chứng (Logs, Checkpoints, Telemetry, Reports)
import os, shutil, json, time

_archive_dir = '/kaggle/working/stair6_v1_artifacts'
os.makedirs(_archive_dir, exist_ok=True)

# 1. Thu thập logs, reports và runs
_target_dirs = [
    '/kaggle/working/logs/stair6_v1',
    '/kaggle/working/reports',
    '/kaggle/working/runs/stair6_v1',
    os.path.expanduser('~/logs/stair6_v1'),
    os.path.expanduser('~/reports'),
]
for _src in _target_dirs:
    if os.path.exists(_src):
        _dst = os.path.join(_archive_dir, os.path.basename(_src))
        try:
            shutil.copytree(_src, _dst, dirs_exist_ok=True)
        except Exception:
            pass

# 2. Tạo manifest tổng thể cho phiên thử nghiệm
_manifest_data = {
    'run_id': RUN_ID,
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    'framework': 'STAIR6-v1 (LCAE-WMSG-CSE)',
    'results': {k: v.get('metrics', {}) for k, v in STAIR6_V1_RESULTS.items()},
    'configs': {k: {cfg_k: str(cfg_v) for cfg_k, cfg_v in v.items()} for k, v in STAIR6_V1_CONFIGS.items()},
}
with open(os.path.join(_archive_dir, 'run_manifest.json'), 'w', encoding='utf-8') as f:
    json.dump(_manifest_data, f, indent=2, ensure_ascii=False)

# 3. Tạo file nén tar.gz và zip trong /kaggle/working
_tar_path = '/kaggle/working/stair6_v1_artifacts.tar.gz'
_zip_path = '/kaggle/working/stair6_v1_artifacts.zip'

print("📦 Đang nén toàn bộ artifacts...")
try:
    subprocess.run(['tar', '-czf', _tar_path, '-C', '/kaggle/working', 'stair6_v1_artifacts'], check=True)
    print(f"✅ Đã tạo tệp tin nén Tarball: {_tar_path} ({os.path.getsize(_tar_path)/(1024**2):.2f} MB)")
except Exception as _e:
    print(f"⚠️ Tarball nén thất bại: {_e}")

try:
    shutil.make_archive('/kaggle/working/stair6_v1_artifacts', 'zip', '/kaggle/working', 'stair6_v1_artifacts')
    print(f"✅ Đã tạo tệp tin nén Zip: {_zip_path} ({os.path.getsize(_zip_path)/(1024**2):.2f} MB)")
except Exception as _e:
    print(f"⚠️ Zip nén thất bại: {_e}")

print("=" * 80)
print("🎉 TOÀN BỘ MINH CHỨNG VÀ KẾT QUẢ THỰC NGHIỆM ĐÃ ĐƯỢC ĐÓNG GÓI THÀNH CÔNG!")
print("=" * 80)
